# 02 - Transformação e Harmonização dos Dados | Camada Silver

Este notebook realiza o tratamento, a harmonização e o enriquecimento dos microdados da PNAD Contínua de 2022 e 2024 armazenados na Camada Bronze.

A Camada Silver tem como objetivo transformar os dados brutos em uma base estruturada e comparável entre os períodos analisados, preservando a granularidade dos registros individuais da pesquisa e as informações necessárias para a realização de estimativas populacionais.

Como os microdados utilizados possuem estrutura de largura fixa e apresentam diferenças na posição de algumas variáveis entre 2022 e 2024, a construção da Silver envolve a identificação das posições correspondentes no dicionário da PNAD, a extração das variáveis selecionadas e sua posterior harmonização.

## Objetivos desta etapa

- extrair dos arquivos brutos as variáveis necessárias ao projeto;
- harmonizar as estruturas dos microdados de 2022 e 2024;
- padronizar tipos e categorias das variáveis selecionadas;
- preservar o peso amostral necessário às estimativas populacionais;
- construir variáveis derivadas utilizadas na identificação dos trabalhadores por plataformas;
- identificar analiticamente os entregadores por aplicativo;
- integrar os dois períodos em uma única estrutura;
- validar os resultados das transformações realizadas;
- persistir a base harmonizada em formato Delta para utilização pela Camada Gold.

## Critério analítico

A identificação dos trabalhadores por plataformas considera a variável derivada `SD14001`, reconstruída a partir das regras presentes na documentação da PNAD para os períodos analisados.

Para a identificação dos entregadores por aplicativo utilizados nas análises posteriores, o projeto distingue a declaração de realização de serviços de entrega por aplicativo (`S140093`) da classificação como trabalhador plataformizado no trabalho principal (`SD14001`).

A população analítica de entregadores é identificada pela combinação dessas condições, registrada na variável `entregador_plataformizado`.

Essa distinção permite evitar que a simples declaração de utilização de aplicativo de entrega seja automaticamente interpretada como pertencimento à população analítica de trabalhadores plataformizados considerada no projeto.

## Peso amostral

Por se tratar de uma pesquisa amostral, o número de registros presentes nos microdados não corresponde diretamente ao número de trabalhadores na população.

Por esse motivo, a variável de peso amostral é preservada na Camada Silver e utilizada nas estimativas populacionais. O projeto mantém, assim, a distinção entre o número de registros amostrais e as estimativas ponderadas produzidas a partir da PNAD Contínua.

Ao final desta etapa, a base harmonizada é persistida como tabela Delta `silver_entregadores_pnad`, que constitui a fonte para a construção da Camada Gold.

In [0]:
# ============================================================
# PROJETO: Observatório do Trabalho por Plataformas
# ETAPA: Transformação dos microdados PNAD - Camada Silver
# ============================================================

from pyspark.sql.functions import (
    col,
    substring,
    trim,
    when,
    lit
)

# Arquivos brutos já validados na camada Bronze

arquivo_2022 = (
    "/Volumes/workspace/bronze/pnad_raw/2022/"
    "extracted/PNADC_2022_trimestre4.txt"
)

arquivo_2024 = (
    "/Volumes/workspace/bronze/pnad_raw/2024/"
    "extracted/PNADC_2024_trimestre3.txt"
)

# Leitura dos arquivos de largura fixa

df_raw_2022 = spark.read.text(arquivo_2022)
df_raw_2024 = spark.read.text(arquivo_2024)

print("Registros brutos - 2022:", df_raw_2022.count())
print("Registros brutos - 2024:", df_raw_2024.count())

In [0]:
# ============================================================
# MAPA DE VARIÁVEIS DA PNAD POR EDIÇÃO
# posição inicial e tamanho conforme dicionários oficiais
# ============================================================

mapa_variaveis = {

    2022: {
        "V1028": (50, 15),
        "V4012": (156, 1),
        "V40121": (157, 1),
        "V4013": (158, 5),

        "sexo": (95, 1),
        "idade": (104, 3),
        "cor_raca": (107, 1),

        "S140091": (1081, 1),
        "S140092": (1082, 1),
        "S140093": (1083, 1),
        "S140094": (1084, 1),
    },

    2024: {
        "V1028": (50, 15),
        "V4012": (156, 1),
        "V40121": (157, 1),
        "V4013": (158, 5),

        "sexo": (95, 1),
        "idade": (104, 3),
        "cor_raca": (107, 1),

        "S140091": (682, 1),
        "S140092": (683, 1),
        "S140093": (684, 1),
        "S140093A": (685, 1),
        "S140094": (686, 1),
    }
}

print("Variáveis mapeadas - 2022:")
for variavel, especificacao in mapa_variaveis[2022].items():
    print(variavel, especificacao)

print("\nVariáveis mapeadas - 2024:")
for variavel, especificacao in mapa_variaveis[2024].items():
    print(variavel, especificacao)

In [0]:
# ============================================================
# FUNÇÃO DE EXTRAÇÃO DAS VARIÁVEIS POR LAYOUT
# ============================================================

from pyspark.sql.functions import col, substring, trim, lit

def extrair_variaveis(df_raw, ano, mapa):
    """
    Extrai variáveis de um arquivo fixed-width da PNAD
    utilizando o mapa de posições correspondente ao ano.
    """

    colunas = []

    for variavel, (posicao, tamanho) in mapa[ano].items():
        colunas.append(
            trim(
                substring(
                    col("value"),
                    posicao,
                    tamanho
                )
            ).alias(variavel)
        )

    return (
        df_raw
        .select(*colunas)
        .withColumn("ano_pnad", lit(ano))
    )


# Aplicação da função aos dois anos

df_extraido_2022 = extrair_variaveis(
    df_raw_2022,
    2022,
    mapa_variaveis
)

df_extraido_2024 = extrair_variaveis(
    df_raw_2024,
    2024,
    mapa_variaveis
)

print("Extração Silver - 2022:")
display(df_extraido_2022.limit(10))

print("Extração Silver - 2024:")
display(df_extraido_2024.limit(10))

In [0]:
# ============================================================
# HARMONIZAÇÃO DOS SCHEMAS - 2022 E 2024
# ============================================================

from pyspark.sql.functions import lit
from pyspark.sql.types import StringType

# S140093A existe em 2024, mas não em 2022.
# Criamos a coluna vazia em 2022 para preservar um schema comum.

df_harmonizado_2022 = (
    df_extraido_2022
    .withColumn("S140093A", lit(None).cast(StringType()))
)

df_harmonizado_2024 = df_extraido_2024


# Ordem padronizada das colunas

colunas_padrao = [
    "ano_pnad",
    "V1028",
    "sexo",
    "idade",
    "cor_raca",
    "V4012",
    "V40121",
    "V4013",
    "S140091",
    "S140092",
    "S140093",
    "S140093A",
    "S140094"
]


df_harmonizado_2022 = df_harmonizado_2022.select(*colunas_padrao)
df_harmonizado_2024 = df_harmonizado_2024.select(*colunas_padrao)


print("Schema harmonizado - 2022:")
df_harmonizado_2022.printSchema()

print("\nSchema harmonizado - 2024:")
df_harmonizado_2024.printSchema()

print("\nAmostra 2022:")
display(df_harmonizado_2022.limit(5))

print("\nAmostra 2024:")
display(df_harmonizado_2024.limit(5))

In [0]:
# ============================================================
# UNIÃO DAS BASES HARMONIZADAS - 2022 E 2024
# ============================================================

from pyspark.sql.functions import col, count

# União por nome das colunas
df_pnad_integrada = (
    df_harmonizado_2022
    .unionByName(df_harmonizado_2024)
)

# ------------------------------------------------------------
# VALIDAÇÃO 1 - Total de registros
# ------------------------------------------------------------

total_2022 = df_harmonizado_2022.count()
total_2024 = df_harmonizado_2024.count()
total_integrado = df_pnad_integrada.count()

print("Registros 2022:", total_2022)
print("Registros 2024:", total_2024)
print("Total esperado:", total_2022 + total_2024)
print("Total após união:", total_integrado)

# ------------------------------------------------------------
# VALIDAÇÃO 2 - Distribuição por ano
# ------------------------------------------------------------

print("\nRegistros por ano:")

display(
    df_pnad_integrada
    .groupBy("ano_pnad")
    .agg(count("*").alias("registros"))
    .orderBy("ano_pnad")
)

# ------------------------------------------------------------
# VALIDAÇÃO 3 - Visualização da base integrada
# ------------------------------------------------------------

print("\nAmostra da base integrada:")

display(
    df_pnad_integrada
    .orderBy("ano_pnad")
    .limit(10)
)

In [0]:
# ============================================================
# TIPAGEM E VARIÁVEIS ANALÍTICAS DA CAMADA SILVER
# ============================================================

from pyspark.sql.functions import col, when

df_silver_base = (
    df_pnad_integrada

    # --------------------------------------------------------
    # Peso amostral
    # --------------------------------------------------------
    .withColumn(
        "peso_amostral",
        col("V1028").cast("double")
    )

    # --------------------------------------------------------
    # Variáveis sociodemográficas
    # --------------------------------------------------------
 .withColumn(
    "idade_anos",
    col("idade").cast("int")
)
.withColumn(
    "sexo_desc",
    when(col("sexo") == "1", "Homem")
    .when(col("sexo") == "2", "Mulher")
    .otherwise("Ignorado")
)

    .withColumn(
        "cor_raca_desc",
        when(col("cor_raca") == "1", "Branca")
        .when(col("cor_raca") == "2", "Preta")
        .when(col("cor_raca") == "3", "Amarela")
        .when(col("cor_raca") == "4", "Parda")
        .when(col("cor_raca") == "5", "Indígena")
        .when(col("cor_raca") == "9", "Ignorado")
        .otherwise("Ignorado")
    )

    # --------------------------------------------------------
    # Indicador: declarou trabalhar com aplicativo de entrega
    # --------------------------------------------------------
    .withColumn(
        "entregador_app",
        when(col("S140093") == "1", 1)
        .when(col("S140093") == "2", 0)
    )

    # --------------------------------------------------------
    # Outros tipos de plataformas
    # --------------------------------------------------------
    .withColumn(
        "app_taxi",
        when(col("S140091") == "1", 1)
        .when(col("S140091") == "2", 0)
    )

    .withColumn(
        "app_transporte_passageiros",
        when(col("S140092") == "1", 1)
        .when(col("S140092") == "2", 0)
    )

    .withColumn(
        "app_servicos",
        when(col("S140094") == "1", 1)
        .when(col("S140094") == "2", 0)
    )
)

print("Schema atualizado da Silver:")
df_silver_base.printSchema()

print("\nAmostra:")
display(
    df_silver_base.select(
        "ano_pnad",
        "peso_amostral",
        "sexo",
        "sexo_desc",
        "idade",
        "idade_anos",
        "cor_raca",
        "cor_raca_desc",
        "entregador_app",
        "V4012",
        "V4013"
    ).limit(20)
)

## Reconstrução da variável derivada SD14001

Nesta etapa, é reconstruída a variável derivada `SD14001`, utilizada para identificar trabalhadores plataformizados no trabalho principal nos períodos analisados.

A regra combina respostas sobre o uso de diferentes tipos de plataformas com informações sobre atividade principal e posição na ocupação. Para os serviços de entrega, a classificação considera grupos específicos de atividades econômicas e, em parte dos casos, condições adicionais relacionadas à posição ocupacional.

A reconstrução é aplicada de forma harmonizada aos microdados de 2022 e 2024 e constitui a base para a identificação analítica dos entregadores plataformizados utilizada nas etapas seguintes.

In [0]:
# ============================================================
# SD14001 - TRABALHADOR PLATAFORMIZADO NO TRABALHO PRINCIPAL
# Regra oficial: 4º tri/2022 e 3º tri/2024
# ============================================================

from pyspark.sql.functions import col, when

# Primeiro grupo de atividades:
# basta S140093 = 1 + uma dessas atividades
atividades_grupo_1 = [
    "49030", "49040", "52020", "53002"
]

# Segundo grupo:
# exige também condição sobre posição na ocupação
atividades_grupo_2 = [
    "48020", "48030", "48041", "48042", "48050", "48060",
    "48071", "48072", "48073", "48074", "48075", "48076",
    "48077", "48078", "48079", "48080", "48090", "48100",
    "56011", "56012", "56020"
]


def criar_sd14001_oficial(df):

    # --------------------------------------------------------
    # S140093 = 1 + atividade do primeiro grupo
    # --------------------------------------------------------
    entrega_grupo_1 = (
        (col("S140093") == "1") &
        (col("V4013").isin(atividades_grupo_1))
    )

    # --------------------------------------------------------
    # S140093 = 1 + atividade do segundo grupo
    # + posição na ocupação prevista pelo IBGE
    # --------------------------------------------------------
    entrega_grupo_2 = (
        (col("S140093") == "1") &
        (col("V4013").isin(atividades_grupo_2)) &
        (
            col("V4012").isin(["5", "6"]) |
            (
                (col("V4012") == "7") &
                (col("V40121") == "1")
            )
        )
    )

    # --------------------------------------------------------
    # SD14001 = 1 - SIM
    # --------------------------------------------------------
    regra_sim = (
        (col("S140091") == "1") |
        (col("S140092") == "1") |
        entrega_grupo_1 |
        entrega_grupo_2 |
        (col("S140094") == "1")
    )

    # --------------------------------------------------------
    # Parte específica da regra de NÃO
    # --------------------------------------------------------
    entrega_nao = (
        (col("S140093") == "2") |
        (
            (col("S140093") == "1") &
            (col("V4013").isin(atividades_grupo_2)) &
            (
                col("V4012").isin(["1", "3"]) |
                col("V40121").isin(["2", "3"])
            )
        )
    )

    # --------------------------------------------------------
    # SD14001 = 2 - NÃO
    # --------------------------------------------------------
    regra_nao = (
        (col("S140091") == "2") &
        (col("S140092") == "2") &
        entrega_nao &
        (col("S140094") == "2")
    )

    return (
        df.withColumn(
            "SD14001",
            when(regra_sim, 1)
            .when(regra_nao, 2)
        )
    )


df_silver_sd14001 = criar_sd14001_oficial(df_silver_base)

print("Distribuição de SD14001 por ano:")

display(
    df_silver_sd14001
    .groupBy("ano_pnad", "SD14001")
    .count()
    .orderBy("ano_pnad", "SD14001")
)

In [0]:
# ============================================================
# VALIDAÇÃO PONDERADA FINAL - SD14001
# ============================================================

from pyspark.sql.functions import col, count, sum as spark_sum

validacao_ponderada = (
    df_silver_sd14001
    .filter(col("SD14001") == 1)
    .groupBy("ano_pnad")
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum(
            col("V1028").cast("double")
        ).alias("estimativa_ponderada")
    )
    .orderBy("ano_pnad")
)

display(validacao_ponderada)

In [0]:
# ============================================================
# DIAGNÓSTICO - ENTREGADORES POR APLICATIVO
# S140093 x SD14001
# ============================================================

from pyspark.sql.functions import col, count, sum as spark_sum

diagnostico_entregadores = (
    df_silver_sd14001
    .filter(col("S140093") == "1")
    .groupBy("ano_pnad", "SD14001")
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum(
            col("V1028").cast("double")
        ).alias("estimativa_ponderada")
    )
    .orderBy("ano_pnad", "SD14001")
)

display(diagnostico_entregadores)

In [0]:
# ============================================================
# INDICADOR ANALÍTICO - ENTREGADOR PLATAFORMIZADO
# ============================================================

from pyspark.sql.functions import col, when

df_silver_entregadores = (
    df_silver_sd14001

    # Resposta direta à pergunta sobre aplicativo de entrega
    .withColumn(
        "app_entrega",
        when(col("S140093") == "1", 1)
        .when(col("S140093") == "2", 0)
    )

    # Entregador que também atende à definição de
    # trabalhador plataformizado reconstruída pelo SD14001
    .withColumn(
        "entregador_plataformizado",
        when(
            (col("S140093") == "1") &
            (col("SD14001") == 1),
            1
        )
        .when(
            col("SD14001").isin([1, 2]),
            0
        )
    )
)

print("Validação do indicador de entregadores:")

display(
    df_silver_entregadores
    .filter(col("entregador_plataformizado") == 1)
    .groupBy("ano_pnad")
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum(
            col("V1028").cast("double")
        ).alias("estimativa_ponderada")
    )
    .orderBy("ano_pnad")
)

## Persistência e validação da camada Silver

Após a extração, harmonização e transformação dos microdados, a base consolidada é persistida em formato Delta. A camada Silver preserva as observações da PNAD Contínua dos dois períodos analisados e incorpora as variáveis tipadas e os indicadores necessários às etapas analíticas posteriores.

Após a gravação, são realizadas verificações da estrutura e dos registros persistidos, buscando assegurar a consistência da base que será utilizada na construção da camada Gold.

In [0]:
# ============================================================
# SALVAMENTO DA CAMADA SILVER
# Base analítica de entregadores plataformizados
# ============================================================

tabela_silver = "silver_entregadores_pnad"

(
    df_silver_entregadores
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)

print("Tabela Silver criada com sucesso:")
print(tabela_silver)

In [0]:
# ============================================================
# VALIDAÇÃO DA TABELA SILVER SALVA
# ============================================================

df_teste_silver = spark.table("silver_entregadores_pnad")

print("Total de registros:")
print(df_teste_silver.count())

print("\nSchema:")
df_teste_silver.printSchema()

print("\nRegistros por ano:")
display(
    df_teste_silver
    .groupBy("ano_pnad")
    .count()
    .orderBy("ano_pnad")
)

In [0]:
# ============================================================
# VALIDAÇÃO FINAL DA TABELA SILVER
# ============================================================

from pyspark.sql.functions import col, count, sum as spark_sum

df_teste_silver = spark.table("silver_entregadores_pnad")

print("1. Total geral da Silver:")
print(df_teste_silver.count())

print("\n2. Registros por ano:")
display(
    df_teste_silver
    .groupBy("ano_pnad")
    .agg(count("*").alias("registros"))
    .orderBy("ano_pnad")
)

print("\n3. Entregadores plataformizados por ano:")
display(
    df_teste_silver
    .filter(col("entregador_plataformizado") == 1)
    .groupBy("ano_pnad")
    .agg(
        count("*").alias("registros_amostrais"),
        spark_sum("peso_amostral").alias("estimativa_ponderada")
    )
    .orderBy("ano_pnad")
)